# JurisData Analytics: Fase 4: Pareceres Automáticos com LLM

**Objetivo:** Gerar pareceres jurídicos estruturados usando RAG + Gemini API
**Input:** ml.features_decisao + analytics.processos (contexto histórico)
**Output:** llm.pareceres — pareceres auditáveis com risk_score, fundamentação e recomendação

In [1]:
#Célula 1: Instalações e imports
%pip install groq --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\fabri\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
#Célula 2: Imports e configs
import os
import json
import hashlib
import psycopg2
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

PG_CONFIG = {
    "host":     os.getenv("PG_HOST", "localhost"),
    "port":     int(os.getenv("PG_PORT", "5432")),
    "dbname":   os.getenv("PG_DBNAME", "jurisdata"),
    "user":     os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD", ""),
}

def get_conn():
    return psycopg2.connect(**PG_CONFIG)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Testa
resp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Responda apenas: ok"}],
)
print("Groq:", resp.choices[0].message.content.strip())

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM ml.features_decisao")
        print("PostgreSQL OK — features_decisao:", cur.fetchone()[0])

Groq: ok
PostgreSQL OK — features_decisao: 40154


In [3]:
#Célula 3: Função de busca de contexto (RAG)
def buscar_contexto(assunto_codigo: int, limite: int = 5) -> list[dict]:
    """
    Busca os casos mais representativos de um assunto no banco.
    Retorna lista de dicts com dados do processo para compor o contexto do LLM.
    """
    sql = """
        SELECT
            p.numero_cnj,
            p.data_ajuizamento,
            p.tempo_tramitacao_dias,
            p.orgao_julgador,
            p.grau,
            a.assunto_nome,
            f.cluster_label
        FROM analytics.processos p
        JOIN ml.features_decisao f ON f.process_id = p.process_id
        JOIN analytics.dim_assunto a ON a.assunto_codigo = p.assunto_codigo
        WHERE p.assunto_codigo = %s
          AND f.cluster_id IS NOT NULL
        ORDER BY p.data_ajuizamento DESC
        LIMIT %s
    """
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, (assunto_codigo, limite))
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
    return [dict(zip(cols, row)) for row in rows]


# Testa com Verbas Rescisórias (assunto mais frequente)
contexto = buscar_contexto(assunto_codigo=13858, limite=3)
for c in contexto:
    print(c)

{'numero_cnj': '00011904220245210003', 'data_ajuizamento': datetime.date(2024, 12, 29), 'tempo_tramitacao_dias': 429, 'orgao_julgador': '12ª Vara do Trabalho de Natal', 'grau': 'G1', 'assunto_nome': 'Salário/Diferença Salarial', 'cluster_label': 'Litígios Recentes — Tramitação Ágil'}
{'numero_cnj': '00011556120245210010', 'data_ajuizamento': datetime.date(2024, 12, 29), 'tempo_tramitacao_dias': 474, 'orgao_julgador': '10ª Vara do Trabalho de Natal', 'grau': 'G1', 'assunto_nome': 'Salário/Diferença Salarial', 'cluster_label': 'Litígios Recentes — Tramitação Ágil'}
{'numero_cnj': '00011642120245210043', 'data_ajuizamento': datetime.date(2024, 12, 23), 'tempo_tramitacao_dias': 480, 'orgao_julgador': '13ª Vara do Trabalho de Natal', 'grau': 'G1', 'assunto_nome': 'Salário/Diferença Salarial', 'cluster_label': 'Litígios Recentes — Tramitação Ágil'}


In [4]:
#Célula 4: Função de geração do parecer
def gerar_parecer(assunto_codigo: int, assunto_nome: str, orgao: str = "TRT-21") -> dict:
    """
    Gera um parecer jurídico estruturado usando RAG + LLM.
    Retorna dict com risk_score, fundamentacao e recomendacao.
    """
    contexto = buscar_contexto(assunto_codigo, limite=5)

    contexto_texto = "\n".join([
        f"- Processo {c['numero_cnj']} | Ajuizado em {c['data_ajuizamento']} | "
        f"{c['tempo_tramitacao_dias']} dias de tramitação | "
        f"Órgão: {c['orgao_julgador']} | Perfil: {c['cluster_label']}"
        for c in contexto
    ])

    prompt = f"""Você é um especialista em direito trabalhista brasileiro com foco em jurimetria.
Com base nos casos históricos reais do {orgao} abaixo, gere um parecer de risco para uma nova reclamação trabalhista.

ASSUNTO DA RECLAMAÇÃO: {assunto_nome}
TRIBUNAL: {orgao}

CASOS HISTÓRICOS SIMILARES:
{contexto_texto}

Responda APENAS com um JSON válido, sem texto adicional, sem markdown, sem explicações fora do JSON:
{{
  "assunto": "{assunto_nome}",
  "tribunal": "{orgao}",
  "risk_score": <número de 0 a 100 indicando probabilidade de condenação>,
  "nivel_risco": "<Baixo|Médio|Alto>",
  "tempo_medio_estimado_dias": <número>,
  "fundamentacao": "<2 a 3 frases explicando o risco com base nos casos históricos>",
  "recomendacao": "<1 frase com recomendação prática para o departamento jurídico>"
}}"""

    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=500,
    )

    raw = resp.choices[0].message.content.strip()

    try:
        parecer = json.loads(raw)
    except json.JSONDecodeError:
        # Tenta extrair o JSON se vier com texto extra
        inicio = raw.find("{")
        fim = raw.rfind("}") + 1
        parecer = json.loads(raw[inicio:fim])

    parecer["input_tokens"]  = resp.usage.prompt_tokens
    parecer["output_tokens"] = resp.usage.completion_tokens

    return parecer


# Testa com Verbas Rescisórias
teste = gerar_parecer(assunto_codigo=13858, assunto_nome="Salário/Diferença Salarial")
print(json.dumps(teste, ensure_ascii=False, indent=2))

{
  "assunto": "Salário/Diferença Salarial",
  "tribunal": "TRT-21",
  "risk_score": 80,
  "nivel_risco": "Alto",
  "tempo_medio_estimado_dias": 450,
  "fundamentacao": "Com base nos casos históricos, observa-se que as reclamações trabalhistas relacionadas a salário e diferença salarial no TRT-21 têm uma tramitação ágil, com média de 450 dias. Além disso, os processos recentes apresentam uma tendência de condenação favorável aos reclamantes. Isso sugere um risco alto de condenação para o empregador.",
  "recomendacao": "O departamento jurídico deve priorizar a análise detalhada dos contratos e pagamentos salariais para evitar erros e irregularidades que possam levar a condenações judiciais.",
  "input_tokens": 615,
  "output_tokens": 215
}


In [5]:
#Célula 5: Geração em lote e persistência na llm.pareceres
from psycopg2.extras import Json
import time

# Busca os top 10 assuntos mais frequentes
with get_conn() as conn:
    top_assuntos = pd.read_sql("""
        SELECT assunto_codigo, assunto_nome, total_processos
        FROM analytics.vw_assuntos
        LIMIT 10
    """, conn)

print("Assuntos que receberão pareceres:")
print(top_assuntos.to_string(index=False))

Assuntos que receberão pareceres:
 assunto_codigo                         assunto_nome  total_processos
          13970                   Verbas Rescisórias             4956
          13968                    Rescisão Indireta             2347
          13994                         Aviso Prévio             2146
          13875           Adicional de Insalubridade             1959
          13722 Reconhecimento de Relação de Emprego             1661
          13787            Adicional de Horas Extras             1564
          14000          Multa do Artigo  477 da CLT             1296
          13769                         Horas Extras             1019
          13998                 Multa de 40% do FGTS              965
          13719                                 FGTS              941


C:\Users\fabri\AppData\Local\Temp\ipykernel_30804\3030814256.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top_assuntos = pd.read_sql("""


In [6]:
#Célula 6: criação de tabela de pareceres
DDL_PARECERES = """
CREATE TABLE IF NOT EXISTS llm.pareceres (
    id              BIGSERIAL PRIMARY KEY,
    input_hash      TEXT UNIQUE NOT NULL,
    assunto_codigo  INTEGER,
    assunto_nome    TEXT,
    tribunal        TEXT,
    risk_score      NUMERIC(5,2),
    fundamentacao   TEXT,
    recomendacao    TEXT,
    output_json     JSONB NOT NULL,
    model_name      TEXT,
    input_tokens    INTEGER,
    output_tokens   INTEGER,
    generated_at    TIMESTAMPTZ DEFAULT NOW()
);
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_PARECERES)
    conn.commit()

print("Tabela llm.pareceres OK")

Tabela llm.pareceres OK


In [7]:
#Célula 7: geração em lote e persistência
SQL_INSERT = """
    INSERT INTO llm.pareceres (
        input_hash, assunto_codigo, assunto_nome, tribunal,
        risk_score, fundamentacao, recomendacao, output_json,
        model_name, input_tokens, output_tokens
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (input_hash) DO NOTHING
"""

gerados = 0
erros   = 0

for _, row in top_assuntos.iterrows():
    try:
        parecer = gerar_parecer(
            assunto_codigo=int(row["assunto_codigo"]),
            assunto_nome=row["assunto_nome"],
        )

        input_hash = hashlib.md5(
            f"{row['assunto_codigo']}_TRT-21".encode()
        ).hexdigest()

        with get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(SQL_INSERT, (
                    input_hash,
                    int(row["assunto_codigo"]),
                    row["assunto_nome"],
                    "TRT-21",
                    parecer["risk_score"],
                    parecer["fundamentacao"],
                    parecer["recomendacao"],
                    Json(parecer),
                    "llama-3.3-70b-versatile",
                    parecer["input_tokens"],
                    parecer["output_tokens"],
                ))
            conn.commit()

        gerados += 1
        print(f"OK [{gerados}/10] {row['assunto_nome']} — risk_score: {parecer['risk_score']}")
        time.sleep(1)  # respeita rate limit do Groq

    except Exception as e:
        erros += 1
        print(f"ERRO — {row['assunto_nome']}: {e}")

print(f"\nConcluído: {gerados} pareceres gerados, {erros} erros")

OK [1/10] Verbas Rescisórias — risk_score: 80
OK [2/10] Rescisão Indireta — risk_score: 80
OK [3/10] Aviso Prévio — risk_score: 80
OK [4/10] Adicional de Insalubridade — risk_score: 80
OK [5/10] Reconhecimento de Relação de Emprego — risk_score: 80
OK [6/10] Adicional de Horas Extras — risk_score: 80
OK [7/10] Multa do Artigo  477 da CLT — risk_score: 80
OK [8/10] Horas Extras — risk_score: 80
OK [9/10] Multa de 40% do FGTS — risk_score: 80
OK [10/10] FGTS — risk_score: 80

Concluído: 10 pareceres gerados, 0 erros


In [8]:
#Célula 8: Validação final
with get_conn() as conn:
    print("Pareceres gerados:")
    print(pd.read_sql("""
        SELECT assunto_nome, risk_score, model_name, input_tokens, output_tokens, generated_at
        FROM llm.pareceres
        ORDER BY risk_score DESC
    """, conn).to_string(index=False))

    print("\nConsumo total de tokens:")
    print(pd.read_sql("""
        SELECT
            SUM(input_tokens)  AS total_input,
            SUM(output_tokens) AS total_output,
            SUM(input_tokens + output_tokens) AS total_geral
        FROM llm.pareceres
    """, conn).to_string(index=False))

Pareceres gerados:
                        assunto_nome  risk_score              model_name  input_tokens  output_tokens                     generated_at
                                FGTS        80.0 llama-3.3-70b-versatile           609            212 2026-05-18 17:02:23.095846+00:00
                Multa de 40% do FGTS        80.0 llama-3.3-70b-versatile           611            214 2026-05-18 17:02:21.039281+00:00
                  Verbas Rescisórias        80.0 llama-3.3-70b-versatile           608            215 2026-05-18 17:02:00.844274+00:00
                        Aviso Prévio        80.0 llama-3.3-70b-versatile           608            202 2026-05-18 17:02:06.395885+00:00
          Adicional de Insalubridade        80.0 llama-3.3-70b-versatile           613            232 2026-05-18 17:02:08.941286+00:00
           Adicional de Horas Extras        80.0 llama-3.3-70b-versatile           609            230 2026-05-18 17:02:13.695764+00:00
         Multa do Artigo  477 da CLT

C:\Users\fabri\AppData\Local\Temp\ipykernel_30804\3647574962.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql("""
C:\Users\fabri\AppData\Local\Temp\ipykernel_30804\3647574962.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql("""


In [10]:
with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT assunto_nome, risk_score, fundamentacao, recomendacao
            FROM llm.pareceres
            ORDER BY generated_at
            LIMIT 3
        """)
        for row in cur.fetchall():
            print("Demonstração de Fundamentações e Recomendações\n")
            print("="*60)
            print(f"Assunto:       {row[0]}")
            print(f"Risk Score:    {row[1]}")
            print(f"Fundamentação: {row[2]}")
            print(f"Recomendação:  {row[3]}")

Demonstração de Fundamentações e Recomendações

Assunto:       Verbas Rescisórias
Risk Score:    80.00
Fundamentação: Com base nos casos históricos, observa-se uma tendência de tramitação ágil e decisões favoráveis aos reclamantes, o que aumenta o risco de condenação. Além disso, a média de tramitação de 472 dias indica uma resolução relativamente rápida. Os casos analisados apresentam perfis de litígios recentes com tramitação ágil, o que reforça a probabilidade de condenação.
Recomendação:  O departamento jurídico deve priorizar a negociação de um acordo extrajudicial ou preparar uma defesa sólida para minimizar os danos financeiros.
Demonstração de Fundamentações e Recomendações

Assunto:       Rescisão Indireta
Risk Score:    70.00
Fundamentação: Com base nos casos históricos, observa-se que a tramitação de processos de rescisão indireta no TRT-21 tem sido ágil, com tempos de tramitação variando de 218 a 472 dias. Além disso, a maioria dos processos tem sido julgada em varas do tra